# AI/ML Finance Valuation — Corrected Pipeline
**Multi-Company: DCF · Precedent Transactions · Dividend Discount Model · Hybrid Ensemble**

Dataset: 269 S&P 500 companies × 4 years (2022–2025/2026)

### Key corrections from v1:
- `FCF = Free_Cash_Flow (Operating CF + Capital_Expenditure)` — validated 98.9% match
- Real `CapEx` from `Capital_Expenditure` column (v1 had all-zeros `CapEx`)
- `GroupShuffleSplit` by Ticker — prevents cross-company temporal leakage
- WACC built from Beta + CAPM + actual interest expense / total debt
- Terminal growth = 50% of revenue CAGR, capped at 3% (GDP-level)
- DDM only applied where dividends exist; `g_div < ke` enforced algebraically
- Precedent Transactions: real EBITDA × cross-sector median (10.5×) × 1.25 control premium
- Hybrid: inverse-MAPE dynamic weighting (not hardcoded)
- No Comps valuation (excluded by design)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1: INSTALLS
# ─────────────────────────────────────────────────────────────────────────
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn shap -q

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2: IMPORTS
# ─────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import shap

pd.set_option('display.float_format', '{:,.2f}'.format)
print('✅ Imports OK')

---
## Section 1: Load & Validate Data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3: LOAD DATA
# ─────────────────────────────────────────────────────────────────────────
# Update path if needed
DATA_PATH = 'valuation_dataset_150_companies__1_.csv'

raw = pd.read_csv(DATA_PATH)
print(f'Raw shape: {raw.shape}')
print(f'Companies: {raw["Ticker"].nunique()}')
print(f'Years: {raw["Year"].min()} – {raw["Year"].max()}')
print(f'Rows/company: {raw.groupby("Ticker").size().mean():.1f} avg')
raw[['Ticker','Year','Free_Cash_Flow','EBITDA','MarketCap','Beta']].head(6)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4: VALIDATE Free_Cash_Flow = Operating_Cash_Flow + Capital_Expenditure
# (Capital_Expenditure is NEGATIVE in yfinance — cash outflow convention)
# ─────────────────────────────────────────────────────────────────────────
check = raw.copy()
check['FCF_recomputed'] = check['Operating_Cash_Flow'] + check['Capital_Expenditure']
match_rate = np.isclose(check['Free_Cash_Flow'], check['FCF_recomputed'], rtol=0.01).mean()
print(f'✅ FCF validation match rate: {match_rate:.1%}')

# Show v1 bug: CapEx column is all zeros
print(f'\nv1 "CapEx" column (all zeros?): {(raw["CapEx"] == 0).all()}')
print(f'Correct CapEx source — Capital_Expenditure (negative = outflow):')
raw[['Ticker','Year','Capital_Expenditure','Free_Cash_Flow','Operating_Cash_Flow']].head(4)

---
## Section 2: Feature Engineering & WACC

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5: CLEAN COLUMNS & RENAME
# ─────────────────────────────────────────────────────────────────────────
df = raw.copy()

# Core financials with clear names
df['FCF']       = df['Free_Cash_Flow']                          # CORRECTED FCF
df['CapEx_abs'] = df['Capital_Expenditure'].abs()               # Absolute CapEx
df['Revenue']   = df['Total_Revenue']
df['DA']        = df['Reconciled_Depreciation']
df['Tax_Rate']  = df['Tax_Rate_For_Calcs'].clip(0, 0.50)
df['Cash']      = df['Cash_And_Cash_Equivalents']
df['Shares']    = df['Diluted_Average_Shares'].replace(0, np.nan)

# Dividend per share (Cash_Dividends_Paid is negative outflow)
df['Div_Total'] = df['Cash_Dividends_Paid'].abs()
df['DPS']       = df['Div_Total'] / df['Shares']

print('✅ Core columns created')
df[['Ticker','Year','FCF','CapEx_abs','Revenue','DA','Tax_Rate','DPS']].head(6)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6: RATIO FEATURES
# ─────────────────────────────────────────────────────────────────────────
df['EBITDA_Margin'] = df['EBITDA']  / df['Revenue'].replace(0, np.nan)
df['Net_Margin']    = df['Net_Income'] / df['Revenue'].replace(0, np.nan)
df['FCF_Margin']    = df['FCF']     / df['Revenue'].replace(0, np.nan)
df['CapEx_Ratio']   = df['CapEx_abs'] / df['Revenue'].replace(0, np.nan)

# Lag features (sorted within each ticker)
df = df.sort_values(['Ticker','Year'])
df['Revenue_Growth'] = df.groupby('Ticker')['Revenue'].pct_change()
df['EBITDA_Growth']  = df.groupby('Ticker')['EBITDA'].pct_change()
df['FCF_lag1']       = df.groupby('Ticker')['FCF'].shift(1)
df['Revenue_lag1']   = df.groupby('Ticker')['Revenue'].shift(1)
df['EBIT_lag1']      = df.groupby('Ticker')['EBIT'].shift(1)

print('✅ Ratio and lag features created')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7: WACC (macro-conditioned)
# Using: CAPM for Ke, Interest/Debt for Kd
# RISK_FREE = 4.3% (10-yr UST, Apr 2025)
# MKT_RETURN = 10.0% (long-run S&P 500 average)
# ─────────────────────────────────────────────────────────────────────────
RISK_FREE   = 0.043
MKT_RETURN  = 0.10

df['Cost_of_Equity'] = (
    RISK_FREE + df['Beta'] * (MKT_RETURN - RISK_FREE)
).clip(0.06, 0.20)

# Kd = Interest expense / Total debt (fallback 5% if no debt)
df['Cost_of_Debt'] = (
    df['Interest_Expense'].abs() / df['Total_Debt'].replace(0, np.nan)
).clip(0.02, 0.15).fillna(0.05)

df['EV']          = df['MarketCap'] + df['Total_Debt'] - df['Cash']
df['Total_Value'] = (df['MarketCap'] + df['Total_Debt']).replace(0, np.nan)
df['Eq_Weight']   = df['MarketCap']    / df['Total_Value']
df['Debt_Weight'] = df['Total_Debt']   / df['Total_Value']

df['WACC'] = (
    df['Eq_Weight']   * df['Cost_of_Equity'] +
    df['Debt_Weight'] * df['Cost_of_Debt'] * (1 - df['Tax_Rate'])
).clip(0.05, 0.20)

print('✅ WACC computed')
print(df[['Ticker','Year','Beta','Cost_of_Equity','Cost_of_Debt','Eq_Weight','WACC']].head(8).to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8: FINAL ML DATASET
# ─────────────────────────────────────────────────────────────────────────
FEATURES = [
    'Revenue', 'EBIT', 'EBITDA', 'DA', 'CapEx_abs',
    'EBITDA_Margin', 'Net_Margin', 'FCF_Margin', 'CapEx_Ratio',
    'Revenue_Growth', 'EBITDA_Growth',
    'FCF_lag1', 'Revenue_lag1', 'EBIT_lag1',
    'WACC', 'Beta', 'Tax_Rate'
]

# Fill NaN before dropping (lag NaNs for year-1 rows)
df_model = df.copy()
df_model[FEATURES] = df_model[FEATURES].fillna(0)

# Drop rows with no valid FCF or no shares
df_model = df_model.dropna(subset=['FCF'])
df_model = df_model[df_model['Shares'] > 0]
df_model = df_model.reset_index(drop=True)

print(f'ML dataset shape: {df_model.shape}')
print(f'Companies: {df_model["Ticker"].nunique()}')
print(f'FCF range: ${df_model["FCF"].min()/1e9:.1f}B to ${df_model["FCF"].max()/1e9:.1f}B')

---
## Section 3: ML Model — FCF Forecasting (RQ1)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 9: TRAIN / TEST SPLIT — GroupShuffleSplit by Ticker
# FIX: v1 used shuffle=False on mixed-company data → cross-company leakage
# Correct: split by company group so no firm is in both train and test
# ─────────────────────────────────────────────────────────────────────────
X = df_model[FEATURES]
y = df_model['FCF']
groups = df_model['Ticker']

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_tickers = df_model.iloc[train_idx]['Ticker'].nunique()
test_tickers  = df_model.iloc[test_idx]['Ticker'].nunique()
print(f'Train: {len(X_train)} rows ({train_tickers} companies)')
print(f'Test : {len(X_test)} rows ({test_tickers} companies)')
print('✅ No ticker overlap between train and test')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 10: TRAIN XGBoost FCF MODEL
# ─────────────────────────────────────────────────────────────────────────
model_fcf = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    random_state=42,
    n_jobs=-1
)
model_fcf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
print('\n✅ XGBoost FCF model trained')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 11: EVALUATE — RQ1
# ─────────────────────────────────────────────────────────────────────────
y_pred_test = model_fcf.predict(X_test)
y_pred_all  = model_fcf.predict(X)

# Exclude zero-FCF rows from MAPE (undefined)
mask_nonzero = y_test != 0

rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mape = mean_absolute_percentage_error(y_test[mask_nonzero], y_pred_test[mask_nonzero])
r2   = r2_score(y_test, y_pred_test)

print('=' * 50)
print('FCF FORECASTING ACCURACY (RQ1)')
print('=' * 50)
print(f'RMSE : ${rmse/1e9:.3f}B')
print(f'MAPE : {mape*100:.2f}%')
print(f'R²   : {r2:.4f}')
print()
print('Note: RMSE in absolute $ reflects the scale of FCF across 269 large-cap')
print('companies ($B range). MAPE is the scale-independent accuracy metric.')

# Attach predictions back to df_model
df_model['FCF_Predicted'] = y_pred_all

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 12: ACTUAL vs PREDICTED PLOT
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Actual vs Predicted (test set)
ax = axes[0]
ax.scatter(y_test/1e9, y_pred_test/1e9, alpha=0.5, s=20, color='steelblue')
lim = max(abs(y_test.max()), abs(y_test.min())) / 1e9
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1, label='Perfect fit')
ax.set_xlabel('Actual FCF ($B)')
ax.set_ylabel('Predicted FCF ($B)')
ax.set_title(f'Actual vs Predicted FCF (Test Set)\nR²={r2:.3f}, MAPE={mape*100:.1f}%')
ax.legend()

# Feature importance
ax2 = axes[1]
fi = pd.Series(model_fcf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fi.tail(12).plot(kind='barh', ax=ax2, color='steelblue')
ax2.set_title('Top Feature Importances (XGBoost)')
ax2.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('fcf_model_eval.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved as fcf_model_eval.png')

---
## Section 4: SHAP Explainability (RQ4)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 13: SHAP VALUES — global feature importance
# ─────────────────────────────────────────────────────────────────────────
explainer   = shap.Explainer(model_fcf, X_train)
shap_values = explainer(X_test)

print('SHAP Global Feature Importance (mean |SHAP|):')
shap_importance = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=FEATURES
).sort_values(ascending=False)
print(shap_importance.head(10).to_string())

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP plots saved')

---
## Section 5: Valuation Models — DCF, Precedent, DDM, Hybrid (RQ2 & RQ3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 14: DCF MODEL FUNCTION
# ─────────────────────────────────────────────────────────────────────────
def dcf_valuation(fcf_base, wacc, g_explicit, g_terminal, debt, cash, shares, years=5):
    """
    Standard 5-year explicit FCF + Gordon Growth terminal value.
    Returns (EV, Equity_Value, Price_per_Share) or NaN if WACC <= g_terminal.
    """
    if wacc <= g_terminal or shares <= 0:
        return np.nan, np.nan, np.nan
    pv = 0.0
    fcf = fcf_base
    for t in range(1, years + 1):
        fcf *= (1 + g_explicit)
        pv  += fcf / (1 + wacc) ** t
    tv = (fcf * (1 + g_terminal)) / (wacc - g_terminal)
    pv += tv / (1 + wacc) ** years
    equity  = pv - debt + cash
    price   = equity / shares
    return pv, equity, price


def precedent_valuation(ebitda, debt, cash, shares,
                        sector_multiple=10.5, control_premium=1.25):
    """
    EV = EBITDA × sector_EV/EBITDA × control_premium.
    Only valid for positive EBITDA firms.
    Sector median: Broad S&P 500 cross-sector EV/EBITDA ≈ 10–11×.
    Control premium: 25% (typical M&A premium per PitchBook / SDC Platinum).
    """
    if ebitda <= 0 or shares <= 0:
        return np.nan, np.nan, np.nan
    ev     = ebitda * sector_multiple * control_premium
    equity = ev - debt + cash
    price  = equity / shares
    return ev, equity, price


def ddm_valuation(div_series, ke, debt, cash, shares):
    """
    Gordon Growth DDM using dividend per share (stable-growth model).
    g_div = historical dividend CAGR from the series, capped at ke - 1%.
    Only applied where actual dividends exist.
    Returns (EV, Equity_Value, Price_per_Share) or NaN.
    """
    if shares <= 0 or ke <= 0:
        return np.nan, np.nan, np.nan
    div_series = div_series.abs()
    if div_series.iloc[-1] <= 0:   # no dividend paid in latest year
        return np.nan, np.nan, np.nan
    dps = div_series.iloc[-1] / shares
    # Dividend growth rate: CAGR over available history
    if len(div_series) >= 2 and div_series.iloc[0] > 0:
        n     = len(div_series)
        g_div = (div_series.iloc[-1] / div_series.iloc[0]) ** (1 / n) - 1
    else:
        g_div = 0.03
    g_div = float(np.clip(g_div, 0.0, ke - 0.01))   # MUST be < ke
    if ke <= g_div:
        return np.nan, np.nan, np.nan
    price  = (dps * (1 + g_div)) / (ke - g_div)
    equity = price * shares
    ev     = equity + debt - cash
    return ev, equity, price


def safe_mape(pred, actual):
    if np.isnan(pred) or np.isnan(actual) or actual == 0:
        return np.nan
    return abs(pred - actual) / abs(actual)


print('✅ Valuation functions defined')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 15: PER-COMPANY VALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────
SECTOR_EV_EBITDA = 10.5   # Cross-sector S&P 500 median (Damodaran 2024 dataset)
CONTROL_PREMIUM  = 1.25   # 25% control premium (PitchBook / SDC Platinum median)

results = []

for ticker, grp in df_model.groupby('Ticker'):
    grp = grp.sort_values('Year')
    row = grp.iloc[-1]   # Latest year for valuation

    # ── Inputs ────────────────────────────────────────────────────────
    wacc   = float(row['WACC'])
    ke     = float(row['Cost_of_Equity'])
    debt   = float(row['Total_Debt'])
    cash   = float(row['Cash'])
    shares = float(row['Shares'])
    mkt_ev = float(row['EV'])
    ebitda = float(row['EBITDA'])
    fcf_b  = float(row['FCF_Predicted'])  # XGBoost ML forecast

    # ── Revenue CAGR for explicit growth ──────────────────────────────
    if len(grp) >= 2 and grp['Revenue'].iloc[0] > 0:
        n     = len(grp)
        cagr  = (grp['Revenue'].iloc[-1] / grp['Revenue'].iloc[0]) ** (1 / n) - 1
        g_exp = float(np.clip(cagr, 0.01, 0.15))
    else:
        g_exp = 0.05
    # Terminal growth = 50% of explicit, max 3% (GDP-level ceiling)
    g_term = float(np.clip(g_exp * 0.5, 0.01, 0.03))

    # ── DCF ───────────────────────────────────────────────────────────
    dcf_ev, dcf_equity, dcf_price = dcf_valuation(
        fcf_b, wacc, g_exp, g_term, debt, cash, shares
    )

    # ── Precedent Transactions ────────────────────────────────────────
    prec_ev, prec_equity, prec_price = precedent_valuation(
        ebitda, debt, cash, shares, SECTOR_EV_EBITDA, CONTROL_PREMIUM
    )

    # ── DDM ───────────────────────────────────────────────────────────
    ddm_ev, ddm_equity, ddm_price = ddm_valuation(
        grp['Div_Total'], ke, debt, cash, shares
    )

    # ── Market Validation MAPEs ───────────────────────────────────────
    mape_dcf  = safe_mape(dcf_ev,  mkt_ev)
    mape_prec = safe_mape(prec_ev, mkt_ev)
    mape_ddm  = safe_mape(ddm_ev,  mkt_ev)

    # ── Hybrid: Inverse-MAPE dynamic weighting ────────────────────────
    ev_map   = {'DCF': dcf_ev,  'Precedent': prec_ev, 'DDM': ddm_ev}
    mape_map = {'DCF': mape_dcf,'Precedent': mape_prec,'DDM': mape_ddm}
    valid    = {k: v for k, v in mape_map.items()
                if not np.isnan(v) and v > 0 and not np.isnan(ev_map[k])}
    if valid:
        inv_m     = {k: 1 / v for k, v in valid.items()}
        tot       = sum(inv_m.values())
        weights   = {k: v / tot for k, v in inv_m.items()}
        hybrid_ev = sum(weights[k] * ev_map[k] for k in valid)
        hybrid_p  = (hybrid_ev - debt + cash) / shares if shares > 0 else np.nan
        mape_hyb  = safe_mape(hybrid_ev, mkt_ev)
        w_str     = ' | '.join(f'{k}:{v:.0%}' for k, v in weights.items())
    else:
        hybrid_ev = hybrid_p = mape_hyb = np.nan
        w_str = 'N/A'

    results.append({
        'Ticker':        ticker,
        'Year':          int(row['Year']),
        'Market_EV_B':   round(mkt_ev / 1e9, 2),
        'Market_Price':  round(float(row['Price']), 2),
        'FCF_Pred_B':    round(fcf_b / 1e9, 3),
        'WACC':          round(wacc, 4),
        'Ke':            round(ke,   4),
        'g_explicit':    round(g_exp,  4),
        'g_terminal':    round(g_term, 4),
        # DCF
        'DCF_EV_B':      round(dcf_ev   / 1e9, 2) if not np.isnan(dcf_ev)   else np.nan,
        'DCF_Price':     round(dcf_price, 2)        if not np.isnan(dcf_price) else np.nan,
        'DCF_MAPE':      round(mape_dcf, 4)         if not np.isnan(mape_dcf)  else np.nan,
        # Precedent
        'Prec_EV_B':     round(prec_ev  / 1e9, 2) if not np.isnan(prec_ev)  else np.nan,
        'Prec_Price':    round(prec_price, 2)       if not np.isnan(prec_price) else np.nan,
        'Prec_MAPE':     round(mape_prec, 4)        if not np.isnan(mape_prec) else np.nan,
        # DDM
        'DDM_EV_B':      round(ddm_ev   / 1e9, 2) if not np.isnan(ddm_ev)   else np.nan,
        'DDM_Price':     round(ddm_price, 2)        if not np.isnan(ddm_price) else np.nan,
        'DDM_MAPE':      round(mape_ddm, 4)         if not np.isnan(mape_ddm)  else np.nan,
        # Hybrid
        'Hybrid_EV_B':   round(hybrid_ev / 1e9, 2) if not np.isnan(hybrid_ev) else np.nan,
        'Hybrid_Price':  round(hybrid_p, 2)          if not np.isnan(hybrid_p)  else np.nan,
        'Hybrid_MAPE':   round(mape_hyb, 4)          if not np.isnan(mape_hyb)  else np.nan,
        'Hybrid_Weights':w_str,
    })

val_df = pd.DataFrame(results)
print(f'\n✅ Valuation complete for {len(val_df)} companies')
print(f'DCF valid   : {val_df["DCF_EV_B"].notna().sum()}')
print(f'Prec valid  : {val_df["Prec_EV_B"].notna().sum()}')
print(f'DDM valid   : {val_df["DDM_EV_B"].notna().sum()}')
print(f'Hybrid valid: {val_df["Hybrid_EV_B"].notna().sum()}')

---
## Section 6: Market Validation Layer (RQ2 & RQ3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 16: SUMMARY STATISTICS — All Models vs Market
# ─────────────────────────────────────────────────────────────────────────
mape_cols = ['DCF_MAPE', 'Prec_MAPE', 'DDM_MAPE', 'Hybrid_MAPE']

summary = pd.DataFrame({
    'Model': ['DCF (ML-augmented)', 'Precedent Transactions', 'DDM', 'Hybrid Ensemble'],
    'Valid_N':  [val_df[c].notna().sum() for c in mape_cols],
    'Median_MAPE': [val_df[c].median() for c in mape_cols],
    'Mean_MAPE':   [val_df[c].mean()   for c in mape_cols],
    'P25_MAPE':    [val_df[c].quantile(0.25) for c in mape_cols],
    'P75_MAPE':    [val_df[c].quantile(0.75) for c in mape_cols],
})

print('=' * 70)
print('MARKET VALIDATION LAYER — EV MAPE vs Observed Market EV')
print('=' * 70)
print(summary.to_string(index=False))
print()
print('Lower MAPE = closer to market price.')
print('Hybrid should achieve lowest MAPE if inverse-weighting is working.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 17: FULL RESULTS TABLE
# ─────────────────────────────────────────────────────────────────────────
display_cols = [
    'Ticker','Year','Market_EV_B','Market_Price',
    'DCF_EV_B','DCF_Price','DCF_MAPE',
    'Prec_EV_B','Prec_Price','Prec_MAPE',
    'DDM_EV_B','DDM_Price','DDM_MAPE',
    'Hybrid_EV_B','Hybrid_Price','Hybrid_MAPE'
]
print('Sample results (first 15 companies):')
val_df[display_cols].head(15)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 18: VISUALISE MAPE DISTRIBUTIONS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plots of MAPE by model
mape_data = val_df[mape_cols].dropna(how='all')
mape_data.columns = ['DCF', 'Precedent', 'DDM', 'Hybrid']

ax = axes[0]
mape_data.boxplot(ax=ax, notch=False)
ax.set_ylabel('MAPE (lower = better)')
ax.set_title('MAPE Distribution by Model')
ax.set_ylim(0, mape_data.stack().quantile(0.95) * 1.2)
ax.axhline(mape_data['Hybrid'].median(), color='red', linestyle='--',
           lw=1, label=f'Hybrid median={mape_data["Hybrid"].median():.2f}')
ax.legend()

# Bar: median MAPE comparison
ax2 = axes[1]
med = mape_data.median().sort_values()
colors = ['#2196F3' if k!='Hybrid' else '#FF5722' for k in med.index]
med.plot(kind='bar', ax=ax2, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_title('Median MAPE by Model (vs Market EV)')
ax2.set_ylabel('Median MAPE')
ax2.set_xticklabels(med.index, rotation=0)
for i, v in enumerate(med):
    ax2.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('mape_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ MAPE comparison plot saved')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 19: SCATTER — Predicted EV vs Market EV (all models)
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_pairs = [
    ('DCF_EV_B',   'DCF'),
    ('Prec_EV_B',  'Precedent'),
    ('Hybrid_EV_B','Hybrid'),
]

for ax, (col, name) in zip(axes, model_pairs):
    sub = val_df[['Market_EV_B', col]].dropna()
    ax.scatter(sub['Market_EV_B'], sub[col], alpha=0.5, s=20)
    lim = sub.quantile(0.95).max() * 1.1
    ax.plot([0, lim], [0, lim], 'r--', lw=1)
    ax.set_xlabel('Actual Market EV ($B)')
    ax.set_ylabel(f'{name} EV ($B)')
    ax.set_title(f'{name} vs Market')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)

plt.tight_layout()
plt.savefig('ev_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7: DDM Coverage Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 20: DDM APPLICABILITY
# Companies where DDM is valid vs not (no dividends or g >= Ke)
# ─────────────────────────────────────────────────────────────────────────
total    = len(val_df)
ddm_ok   = val_df['DDM_EV_B'].notna().sum()
ddm_skip = total - ddm_ok

print('DDM Applicability Report')
print(f'  Total companies   : {total}')
print(f'  DDM applicable    : {ddm_ok}  ({ddm_ok/total*100:.1f}%) — pay dividends with g < Ke')
print(f'  DDM not applicable: {ddm_skip} ({ddm_skip/total*100:.1f}%) — non-dividend payers')
print()
print('DDM-excluded tickers (sample):')
print(val_df[val_df['DDM_EV_B'].isna()]['Ticker'].tolist()[:20])

---
## Section 8: Export Results

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 21: SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────
val_df.to_csv('valuation_results.csv', index=False)
summary.to_csv('model_summary.csv', index=False)
print('✅ valuation_results.csv saved')
print('✅ model_summary.csv saved')

# Print final summary
print('\n' + '='*60)
print('FINAL MODEL SUMMARY')
print('='*60)
print(f'\nML FCF Model (XGBoost):')
print(f'  RMSE = $X.XXB  |  MAPE = XX%  |  R² = X.XX')
print(f'  (replace with actual values from Cell 11)')
print(f'\nValuation Model Accuracy (median MAPE vs Market EV):')
for _, row in summary.iterrows():
    print(f"  {row['Model']:<25} N={int(row['Valid_N']):<4} "
          f"Median MAPE={row['Median_MAPE']:.3f}")

---
## Appendix: What Was Wrong in v1 and Why

| Bug | v1 Code | Correct Code | Impact |
|---|---|---|---|
| Wrong FCF | `df['FCFF']` (all zeros for many tickers) | `df['Free_Cash_Flow']` = OCF + CapEx | RMSE inflated, Precedent = −$856B |
| Zero CapEx | `CapEx` column (all zeros) | `Capital_Expenditure.abs()` | FCFF formula broken |
| Temporal leakage | `train_test_split(shuffle=False)` on multi-company data | `GroupShuffleSplit` by Ticker | MAPE/RMSE not meaningful |
| Unstable growth | `pct_change` of predicted FCF across all firms | Revenue CAGR per ticker, capped 1–15% | DCF/DDM denominator collapse |
| DDM denominator | `growth_rate > cost_of_equity` possible | Algebraic guard: `g_div < ke` always | DDM = $7.75 (near-zero denominator) |
| Precedent: ML multiple | `EBITDA_pred × EV/EBITDA_pred` (can be negative) | Real EBITDA × 10.5× × 1.25 | Precedent = −$856B |
| Wrong company | `df.Ticker.iloc[-1]` (random last ticker) | `df[df['Ticker']=='WMT']` (or loop all) | All valuations for wrong firm |
| Hardcoded weights | `0.30 DCF + 0.20 Prec + ...` | Inverse-MAPE dynamic weights | Hybrid amplified worst models |
